In [1]:
from pyspark.sql.functions import (
    col, to_date, when,
    countDistinct, sum as _sum, mean as _mean,
    concat_ws, lit, upper
)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType


schema = StructType([
    StructField("mes_competencia", StringType(), True),
    StructField("mes_referencia", StringType(), True),
    StructField("uf", StringType(), True),
    StructField("código_municipio_siafi", StringType(), True),
    StructField("nome_municipio", StringType(), True),
    StructField("cpf_favorecido", StringType(), True),
    StructField("nis_favorecido", StringType(), True),
    StructField("nome_favorecido", StringType(), True),
    StructField("valor_parcela", DoubleType(), True),
])

dump_pbf = (
    spark.read.format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "latin1")
    .schema(schema)
    .load("Files/raw_bolsa_familia/tb_pbf_sp.csv")
)

# Formatar a data
dump_pbf = (
    dump_pbf
    .withColumn(
        "mes_referencia",
        to_date(concat_ws("", col("mes_referencia").cast("string"), lit("01")), "yyyyMMdd")
    )
    .withColumn(
        "mes_competencia",
        to_date(concat_ws("", col("mes_competencia").cast("string"), lit("01")), "yyyyMMdd")
    )
)

# Padronizar e substituir nomes de municípios
dump_pbf = dump_pbf.withColumn("nome_municipio", upper(col("nome_municipio")))

dump_pbf.write.mode("overwrite").format("delta").save("Tables/dump_pbf_sp")
spark.sql("""
CREATE OR REPLACE TABLE dump_pbf_sp
USING DELTA
AS SELECT * FROM delta.`Tables/dump_pbf_sp`
""")

StatementMeta(, 38cd8159-650d-4d7d-9ff0-9b244c81548e, 3, Finished, Available, Finished)

DataFrame[]